# ScamShield BD — Phase 3B: Pretrained Multilingual Encoders (Kaggle GPU)
**Team Name:** Shield

## Pretrained Model Declarations
| Model | Source | License | Role |
| :--- | :--- | :--- | :--- |
| `intfloat/multilingual-e5-base` | HuggingFace Hub | MIT | Frozen embeddings (`query: ` prefix) + LogisticRegression |
| `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` | HuggingFace Hub | Apache-2.0 | Frozen embeddings + LogisticRegression |
| `xlm-roberta-base` | HuggingFace Hub | MIT | Fine-tuned 3-head multi-task encoder (Tasks A, B, C) |

## Compliance Notes
- Fixed seed `42` across `random`, `numpy`, `torch`, `transformers`.
- No training, MLM, or pseudo-labeling on `test.csv`. `test.csv` is used **only** for the 10,000-row inference throughput benchmark.
- Intermediate embeddings, model checkpoints, and held-out probabilities are cached in `/kaggle/working` for cell restartability.


In [ ]:
import os
import re
import time
import random
import unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, set_seed as hf_set_seed

SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)

seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK_DIR = '/kaggle/working'
os.makedirs(WORK_DIR, exist_ok=True)

INPUT_ROOT = '/kaggle/input'
assert os.path.isdir(INPUT_ROOT), f'{INPUT_ROOT} not found! Are you running on Kaggle?'
input_dirs = os.listdir(INPUT_ROOT)
print('Contents of /kaggle/input:', input_dirs)

# Auto-detect competition data & private synthetic dataset
COMP_DIR = '/kaggle/input/nsucec-neurahack-2026-preliminary'
if not os.path.isdir(COMP_DIR):
    for d in input_dirs:
        cand = os.path.join(INPUT_ROOT, d)
        if os.path.isfile(os.path.join(cand, 'test.csv')) and os.path.isfile(os.path.join(cand, 'sample_submission.csv')):
            COMP_DIR = cand
            break
if not os.path.isfile(os.path.join(COMP_DIR, 'test.csv')):
    raise FileNotFoundError(f'Could not locate competition folder with test.csv in {input_dirs}')

SYNTH_DIR = '/kaggle/input/shield-synth-v2'
if not os.path.isdir(SYNTH_DIR):
    for d in input_dirs:
        cand = os.path.join(INPUT_ROOT, d)
        if os.path.isfile(os.path.join(cand, 'synthetic_v2.csv')):
            SYNTH_DIR = cand
            break
if not os.path.isfile(os.path.join(SYNTH_DIR, 'synthetic_v2.csv')):
    raise FileNotFoundError(f'Could not locate dataset folder with synthetic_v2.csv in {input_dirs}')

print(f'DEVICE={DEVICE} | COMP_DIR={COMP_DIR} | SYNTH_DIR={SYNTH_DIR}')


## 1. Inlined Preprocessing & Noise Augmenter

In [ ]:
import re
import unicodedata

# Common Cyrillic / Greek homoglyphs mapped to Latin equivalents
HOMOGLYPH_MAP = {
    # Cyrillic lowercase
    "\u0430": "a",  # а -> a
    "\u0435": "e",  # е -> e
    "\u0451": "e",  # ё -> e
    "\u0456": "i",  # і -> i
    "\u0457": "i",  # ї -> i
    "\u0458": "j",  # ј -> j
    "\u043e": "o",  # о -> o
    "\u0440": "p",  # р -> p
    "\u0441": "c",  # с -> c
    "\u0443": "y",  # у -> y
    "\u0445": "x",  # х -> x
    "\u0455": "s",  # ѕ -> s
    "\u0501": "d",  # ԁ -> d
    "\u051b": "q",  # ԛ -> q
    "\u051d": "w",  # ԝ -> w
    # Cyrillic uppercase
    "\u0410": "A",  # А -> A
    "\u0412": "B",  # В -> B
    "\u0415": "E",  # Е -> E
    "\u041a": "K",  # К -> K
    "\u041c": "M",  # М -> M
    "\u041d": "H",  # Н -> H
    "\u041e": "O",  # О -> O
    "\u0420": "P",  # Р -> P
    "\u0421": "C",  # С -> C
    "\u0422": "T",  # Т -> T
    "\u0425": "X",  # Х -> X
    "\u04ae": "Y",  # Ү -> Y
    "\u0406": "I",  # І -> I
    # Greek lowercase / uppercase
    "\u03b1": "a",  # α -> a
    "\u03b5": "e",  # ε -> e
    "\u03b9": "i",  # ι -> i
    "\u03bf": "o",  # ο -> o
    "\u03c1": "p",  # ρ -> p
    "\u03c2": "c",  # ς -> c
    "\u0391": "A",  # Α -> A
    "\u0392": "B",  # Β -> B
    "\u0395": "E",  # Ε -> E
    "\u0397": "H",  # Η -> H
    "\u0399": "I",  # Ι -> I
    "\u039a": "K",  # Κ -> K
    "\u039c": "M",  # Μ -> M
    "\u039d": "N",  # Ν -> N
    "\u039f": "O",  # Ο -> O
    "\u03a1": "P",  # Ρ -> P
    "\u03a4": "T",  # Τ -> T
    "\u03a7": "X",  # Χ -> X
}
HOMOGLYPH_TRANS = str.maketrans(HOMOGLYPH_MAP)

# Defanged URL pattern: matches hxxp(s)://..., http(s)://..., or domain with [.]
URL_PATTERN = re.compile(
    r"(?:hxxps?|https?|ftp)://\S+|(?:[a-zA-Z0-9-]+\[\.\])+[a-zA-Z]{2,}(?:/\S*)?",
    re.IGNORECASE,
)

# Digit runs (ASCII and Bengali digits)
DIGIT_RUN_PATTERN = re.compile(r"[0-9\u09e6-\u09ef]+(?:[.,][0-9\u09e6-\u09ef]+)*")

# Latin uppercase letters only (preserves Bangla Unicode untouched)
LATIN_UPPER_PATTERN = re.compile(r"[A-Z]+")


def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. Unicode NFKC normalization
    t = unicodedata.normalize("NFKC", text)

    # 2. Remove only \u200b, \u200c, \ufeff. Keep \u200d (ZWJ for Bangla).
    t = t.replace("\u200b", "").replace("\u200c", "").replace("\ufeff", "")

    # 3. Map common Cyrillic/Greek homoglyphs to Latin
    t = t.translate(HOMOGLYPH_TRANS)

    # 4. Replace defanged URLs with [URL] token
    t = URL_PATTERN.sub(" [URL] ", t)

    # 5. Lowercase Latin text only (leaves Bangla and [URL] handled cleanly)
    t = LATIN_UPPER_PATTERN.sub(lambda m: m.group(0).lower(), t)
    t = t.replace("[url]", "[URL]")

    # 6. Replace digit runs with [NUM] token
    t = DIGIT_RUN_PATTERN.sub(" [NUM] ", t)

    # 7. Preserve "|||" as a spaced separator token and collapse repeated whitespace
    t = t.replace("|||", " ||| ")
    t = re.sub(r"\s+", " ", t).strip()
    return t


def run_unit_tests():
    # Test 1: Homoglyphs (Cyrillic о, а, і, е, р, с)
    s1 = "Cоngratulаtіоns! Рay Tаkа 500 via bKаsh"
    out1 = preprocess_text(s1)
    assert "congratulations!" in out1, f"Failed homoglyph: {out1}"
    assert "pay taka [NUM] via bkash" in out1, f"Failed homoglyph+num: {out1}"

    # Test 2: Zero-width chars (\u200b, \u200c removed; \u200d kept!)
    s2 = "ব\u200cসে Trx\u200bID\ufeff র\u200d্যাংক"
    out2 = preprocess_text(s2)
    assert "\u200b" not in out2 and "\u200c" not in out2 and "\ufeff" not in out2, f"Failed ZW strip: {out2}"
    assert "\u200d" in out2, f"Failed to preserve ZWJ \\u200d: {out2}"
    assert "trxid" in out2, f"Failed Latin lowercase: {out2}"

    # Test 3: Defanged URLs and conversation separator |||
    s3 = "S: Click hxxps://gp-gift[.]xyz/claim ||| R: Okay 12345"
    out3 = preprocess_text(s3)
    assert "[URL]" in out3, f"Failed URL token: {out3}"
    assert "|||" in out3, f"Failed separator preservation: {out3}"
    assert "[NUM]" in out3, f"Failed NUM token: {out3}"

    # Test 4: Bangla text + Bengali digits
    s4 = "আপনার ৫০০ টাকা ক্যাশ আউট সফল হয়েছে।"
    out4 = preprocess_text(s4)
    assert "আপনার [NUM] টাকা ক্যাশ আউট সফল হয়েছে।" == out4, f"Failed Bangla: {out4}"

    print("All preprocess_text unit tests PASSED!")

run_unit_tests()

import re
import random

HOMOGLYPH_SWAPS = {
    "a": "\u0430",  # Cyrillic а
    "e": "\u0435",  # Cyrillic е
    "i": "\u0456",  # Cyrillic і
    "o": "\u043e",  # Cyrillic о
    "p": "\u0440",  # Cyrillic р
    "c": "\u0441",  # Cyrillic с
    "x": "\u0445",  # Cyrillic х
    "y": "\u0443",  # Cyrillic у
}

LEET_MAP = {
    "a": "4",
    "e": "3",
    "i": "1",
    "o": "0",
    "s": "5",
}

EMOJIS = ["📲", "🔥", "⚠️", "🎁", "💰", "✅", "🚨", "💸", "📦", "🎉"]
NOISE_STRINGS = ["wkrjfejejrn", "mnbvcx", "zzqwpo", "xkcdpl", "asdkjh", "qzxvbn", "lkjsdfoiu"]

BANGLISH_VARIANTS = [
    ("bhul", "vul"),
    ("korun", "koren"),
    ("bolen", "bolun"),
    ("pathan", "pathao"),
    ("taka", "tk"),
    ("apnar", "apnr"),
    ("ekhoni", "ekhn e"),
    ("taratari", "tara tari"),
    ("jitechen", "jitsen"),
    ("হয়েছে", "হয়েছে"),
    ("আয়", "আয়"),
    ("করুন", "করেন"),
    ("যোগ", "জোগ"),
]


def _corrupt_word(word: str, rng: random.Random, prob: float) -> str:
    if len(word) <= 2 or word in {"|||", "S:", "R:", "A:", "B:"} or word.startswith("http") or word.startswith("hxxp"):
        return word

    w = word
    # Bangla/Banglish variant
    if rng.random() < prob * 0.5:
        lw = w.lower()
        for src, dst in BANGLISH_VARIANTS:
            if lw == src:
                w = dst
                break
            elif lw == dst:
                w = src
                break

    # Zero-width char (\u200b or \u200c) inside word (works for both Latin & Bangla)
    if len(w) >= 4 and rng.random() < prob * 0.7:
        idx = rng.randint(1, len(w) - 1)
        zw = rng.choice(["\u200b", "\u200c"])
        w = w[:idx] + zw + w[idx:]

    # Latin-specific corruptions
    if re.search(r"[a-zA-Z]", w):
        chars = list(w)
        r = rng.random()
        if r < prob * 0.35 and len(chars) >= 4:
            # Swap two adjacent chars
            idx = rng.randint(1, len(chars) - 2)
            chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]
        elif r < prob * 0.6 and len(chars) >= 4:
            # Drop a char
            idx = rng.randint(1, len(chars) - 2)
            chars.pop(idx)
        elif r < prob * 0.85:
            # Repeat a char
            idx = rng.randint(0, len(chars) - 1)
            chars.insert(idx, chars[idx] * rng.randint(1, 2))
        w = "".join(chars)

        # Leet-speak or homoglyph swap
        if rng.random() < prob * 0.45:
            new_chars = []
            for ch in w:
                lch = ch.lower()
                if lch in HOMOGLYPH_SWAPS and rng.random() < 0.3:
                    new_chars.append(HOMOGLYPH_SWAPS[lch])
                elif lch in LEET_MAP and rng.random() < 0.25:
                    new_chars.append(LEET_MAP[lch])
                else:
                    new_chars.append(ch)
            w = "".join(new_chars)

        # Random capitalization
        if rng.random() < prob * 0.4:
            w = "".join(c.upper() if rng.random() < 0.4 else c.lower() for c in w)
    else:
        # Bangla script character drop / repeat / space split
        if len(w) >= 4 and rng.random() < prob * 0.35:
            idx = rng.randint(1, len(w) - 2)
            if rng.random() < 0.5:
                w = w[:idx] + " " + w[idx:]
            else:
                w = w[:idx] + w[idx + 1:]

    return w


def apply_noise(text: str, rng: random.Random, level: float = 0.3) -> str:
    """Apply realistic adversarial noise to English, Banglish, or Bangla text."""
    if not isinstance(text, str) or not text.strip():
        return text

    # Preserve ||| turns
    turns = text.split("|||")
    noisy_turns = []

    for turn in turns:
        tokens = turn.strip().split(" ")
        new_tokens = []
        for tok in tokens:
            # Defanged URL variation
            if "http" in tok.lower() or "[.]" in tok:
                if rng.random() < level:
                    tok = tok.replace("https://", "hxxps://").replace("http://", "hxxp://")
                    if "[.]" not in tok:
                        tok = tok.replace(".", "[.]")
                new_tokens.append(tok)
                continue

            if rng.random() < level:
                tok = _corrupt_word(tok, rng, prob=min(0.9, level + 0.2))
            new_tokens.append(tok)

            # Occasionally drop space between two tokens
            if len(new_tokens) >= 2 and rng.random() < level * 0.12:
                prev = new_tokens.pop()
                new_tokens[-1] = new_tokens[-1] + prev

        # Insert a random noise string
        if len(new_tokens) >= 3 and rng.random() < level * 0.4:
            pos = rng.randint(1, len(new_tokens) - 1)
            new_tokens.insert(pos, rng.choice(NOISE_STRINGS))

        # Insert an emoji
        if rng.random() < level * 0.5:
            pos = rng.choice([0, len(new_tokens)])
            new_tokens.insert(pos, rng.choice(EMOJIS))

        # Extra spaces occasionally
        joiner = "  " if rng.random() < level * 0.15 else " "
        noisy_turns.append(joiner.join(new_tokens))

    return " ||| ".join(noisy_turns)

## 2. Load Data & Define Reusable Splits (Split 1, Split 2, Split 3)

In [ ]:
TECH_CODES = ['T1', 'T2', 'T3', 'T4', 'T5', 'T6']

def encode_techniques(series):
    y = np.zeros((len(series), len(TECH_CODES)), dtype=int)
    for i, val in enumerate(series.astype(str)):
        parts = val.split('|')
        for j, code in enumerate(TECH_CODES):
            if code in parts:
                y[i, j] = 1
    return y

def postprocess_predictions(labels, tech_probs, actions, confs):
    out_techs = []
    out_actions = actions.copy()
    for i in range(len(labels)):
        if labels[i] == 0:
            out_techs.append('NONE')
            out_actions[i] = 0
        else:
            active = [TECH_CODES[j] for j in range(len(TECH_CODES)) if tech_probs[i, j] >= 0.5]
            if not active:
                active = [TECH_CODES[int(np.argmax(tech_probs[i]))]]
            out_techs.append('|'.join(sorted(active)))
    return labels, out_techs, out_actions, np.round(np.clip(confs, 0.05, 0.98), 4)

def eval_metrics(y_lbl, y_tech, y_act, p_lbl, p_tech_probs, p_act, p_confs):
    p_lbl_pp, p_tech_strs, p_act_pp, _ = postprocess_predictions(p_lbl, p_tech_probs, p_act, p_confs)
    p_tech_bin = encode_techniques(pd.Series(p_tech_strs))
    f1_a = f1_score(y_lbl, p_lbl_pp, average='macro')
    per_cls_a = {k: round(float(v), 4) for k, v in enumerate(f1_score(y_lbl, p_lbl_pp, average=None, labels=[0,1,2,3,4,5], zero_division=0))}
    f1_b = float(np.mean([f1_score(y_tech[:, j], p_tech_bin[:, j], zero_division=0) for j in range(6)]))
    f1_c = f1_score(y_act, p_act_pp, average='macro')
    return f1_a, f1_b, f1_c, per_cls_a

real_path = os.path.join(COMP_DIR, 'sample.csv')
if not os.path.isfile(real_path):
    real_path = os.path.join(SYNTH_DIR, 'sample.csv')
real_df = pd.read_csv(real_path)
synth_df = pd.read_csv(os.path.join(SYNTH_DIR, 'synthetic_v2.csv'), keep_default_na=False)

rng = random.Random(SEED)
real_df['clean_text'] = real_df['text'].fillna('').astype(str).apply(preprocess_text)
real_df['noisy_text'] = [preprocess_text(apply_noise(t, rng, level=0.35)) for t in real_df['text'].fillna('').astype(str)]
synth_df['clean_text'] = synth_df['text'].fillna('').astype(str).apply(preprocess_text)
synth_df['noisy_text'] = [preprocess_text(apply_noise(t, rng, level=0.35)) for t in synth_df['text'].fillna('').astype(str)]

real_lbl = real_df['label'].astype(int).values
real_tech = encode_techniques(real_df['techniques'])
real_act = real_df['action'].astype(int).values
real_ids = real_df['id'].astype(int).values

synth_lbl = synth_df['label'].astype(int).values
synth_tech = encode_techniques(synth_df['techniques'])
synth_act = synth_df['action'].astype(int).values

# Split 1: scenario-held-out (5-fold StratifiedGroupKFold on synthetic rows excluding real_paraphrase)
s1_mask = (synth_df['source'] != 'real_paraphrase').values
s1_indices = np.where(s1_mask)[0]
sgkf = StratifiedGroupKFold(n_splits=5)
split1_folds = list(sgkf.split(s1_indices, synth_lbl[s1_indices], synth_df.loc[s1_mask, 'scenario_id'].values))
split1_folds = [(s1_indices[tr], s1_indices[va]) for tr, va in split1_folds]

# Split 2: real-row check (MAIN METRIC) -> train on synthetic_v2 only, eval on all 86 real rows
split2_train_idx = np.where(synth_df['source'] == 'synthetic_v2')[0]

# Split 3: 5-fold over 86 real rows + train-fold paraphrases + all synthetic scenarios
para_mask = (synth_df['source'] == 'real_paraphrase').values
para_parent_ids = np.full(len(synth_df), -1, dtype=int)
para_parent_ids[para_mask] = synth_df.loc[para_mask, 'scenario_id'].str.replace('real_para_', '', regex=False).astype(int).values
skf_real = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
split3_folds = list(skf_real.split(real_df, real_lbl))

print(f'Real rows: {len(real_df)} | Synth v2 pool: {len(synth_df)} | Split1 synth scenarios: {len(s1_indices)} | Split2 train: {len(split2_train_idx)}')
SUMMARY_RESULTS = []


## 3. Experiment 1: TF-IDF (`char_wb` 2–5 + `word` 1–2) + LogisticRegression

In [ ]:
def fit_predict_tfidf(tr_texts, tr_lbl, tr_tech, tr_act, va_clean, va_noisy, sample_weight=None):
    cv = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True, dtype=np.float32)
    wv = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32)
    X_tr = hstack([cv.fit_transform(tr_texts), wv.fit_transform(tr_texts)]).tocsr()
    X_va_c = hstack([cv.transform(va_clean), wv.transform(va_clean)]).tocsr()
    X_va_n = hstack([cv.transform(va_noisy), wv.transform(va_noisy)]).tocsr()
    ca = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_tr, tr_lbl, sample_weight=sample_weight)
    cb = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)).fit(X_tr, tr_tech)
    cc = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_tr, tr_act, sample_weight=sample_weight)
    return (ca.predict_proba(X_va_c), cb.predict_proba(X_va_c), cc.predict_proba(X_va_c),
            ca.predict_proba(X_va_n), cb.predict_proba(X_va_n), cc.predict_proba(X_va_n))

tfidf_cache = os.path.join(WORK_DIR, 'preds_tfidf.npz')
if os.path.isfile(tfidf_cache):
    d = np.load(tfidf_cache)
    s1_pa_c, s1_pb_c, s1_pc_c, s1_pa_n, s1_pb_n, s1_pc_n = d['s1_pa_c'], d['s1_pb_c'], d['s1_pc_c'], d['s1_pa_n'], d['s1_pb_n'], d['s1_pc_n']
    s2_pa_c, s2_pb_c, s2_pc_c, s2_pa_n, s2_pb_n, s2_pc_n = d['s2_pa_c'], d['s2_pb_c'], d['s2_pc_c'], d['s2_pa_n'], d['s2_pb_n'], d['s2_pc_n']
    s3_pa_c, s3_pb_c, s3_pc_c, s3_pa_n, s3_pb_n, s3_pc_n = d['s3_pa_c'], d['s3_pb_c'], d['s3_pc_c'], d['s3_pa_n'], d['s3_pb_n'], d['s3_pc_n']
    print('Loaded cached TF-IDF predictions from', tfidf_cache)
else:
    # Split 1
    n_s = len(synth_df)
    s1_pa_c, s1_pb_c, s1_pc_c = np.zeros((n_s, 6)), np.zeros((n_s, 6)), np.zeros((n_s, 6))
    s1_pa_n, s1_pb_n, s1_pc_n = np.zeros((n_s, 6)), np.zeros((n_s, 6)), np.zeros((n_s, 6))
    for tr_i, va_i in split1_folds:
        pac, pbc, pcc, pan, pbn, pcn = fit_predict_tfidf(
            synth_df.loc[tr_i, 'clean_text'].tolist(), synth_lbl[tr_i], synth_tech[tr_i], synth_act[tr_i],
            synth_df.loc[va_i, 'clean_text'].tolist(), synth_df.loc[va_i, 'noisy_text'].tolist()
        )
        s1_pa_c[va_i], s1_pb_c[va_i], s1_pc_c[va_i] = pac, pbc, pcc
        s1_pa_n[va_i], s1_pb_n[va_i], s1_pc_n[va_i] = pan, pbn, pcn
    # Split 2
    s2_pa_c, s2_pb_c, s2_pc_c, s2_pa_n, s2_pb_n, s2_pc_n = fit_predict_tfidf(
        synth_df.loc[split2_train_idx, 'clean_text'].tolist(), synth_lbl[split2_train_idx], synth_tech[split2_train_idx], synth_act[split2_train_idx],
        real_df['clean_text'].tolist(), real_df['noisy_text'].tolist()
    )
    # Split 3
    n_r = len(real_df)
    s3_pa_c, s3_pb_c, s3_pc_c = np.zeros((n_r, 6)), np.zeros((n_r, 6)), np.zeros((n_r, 6))
    s3_pa_n, s3_pb_n, s3_pc_n = np.zeros((n_r, 6)), np.zeros((n_r, 6)), np.zeros((n_r, 6))
    for tr_r, va_r in split3_folds:
        tr_pids = set(real_ids[tr_r])
        tr_para_idx = np.where(np.isin(para_parent_ids, list(tr_pids)))[0]
        synth_tr_idx = np.concatenate([s1_indices, tr_para_idx])
        tr_texts = real_df.loc[tr_r, 'clean_text'].tolist() + synth_df.loc[synth_tr_idx, 'clean_text'].tolist()
        tr_l = np.concatenate([real_lbl[tr_r], synth_lbl[synth_tr_idx]])
        tr_t = np.vstack([real_tech[tr_r], synth_tech[synth_tr_idx]])
        tr_a = np.concatenate([real_act[tr_r], synth_act[synth_tr_idx]])
        w = np.concatenate([np.full(len(tr_r), 5.0), np.ones(len(synth_tr_idx))])
        pac, pbc, pcc, pan, pbn, pcn = fit_predict_tfidf(
            tr_texts, tr_l, tr_t, tr_a,
            real_df.loc[va_r, 'clean_text'].tolist(), real_df.loc[va_r, 'noisy_text'].tolist(), sample_weight=w
        )
        s3_pa_c[va_r], s3_pb_c[va_r], s3_pc_c[va_r] = pac, pbc, pcc
        s3_pa_n[va_r], s3_pb_n[va_r], s3_pc_n[va_r] = pan, pbn, pcn
    np.savez(tfidf_cache, s1_pa_c=s1_pa_c, s1_pb_c=s1_pb_c, s1_pc_c=s1_pc_c, s1_pa_n=s1_pa_n, s1_pb_n=s1_pb_n, s1_pc_n=s1_pc_n,
             s2_pa_c=s2_pa_c, s2_pb_c=s2_pb_c, s2_pc_c=s2_pc_c, s2_pa_n=s2_pa_n, s2_pb_n=s2_pb_n, s2_pc_n=s2_pc_n,
             s3_pa_c=s3_pa_c, s3_pb_c=s3_pb_c, s3_pc_c=s3_pc_c, s3_pa_n=s3_pa_n, s3_pb_n=s3_pb_n, s3_pc_n=s3_pc_n)

def report_all_splits(name, s1_c, s1_n, s2_c, s2_n, s3_c, s3_n, throughput='-'):
    m1c = eval_metrics(synth_lbl[s1_indices], synth_tech[s1_indices], synth_act[s1_indices], np.argmax(s1_c[0][s1_indices], 1), s1_c[1][s1_indices], np.argmax(s1_c[2][s1_indices], 1), np.max(s1_c[0][s1_indices], 1))
    m1n = eval_metrics(synth_lbl[s1_indices], synth_tech[s1_indices], synth_act[s1_indices], np.argmax(s1_n[0][s1_indices], 1), s1_n[1][s1_indices], np.argmax(s1_n[2][s1_indices], 1), np.max(s1_n[0][s1_indices], 1))
    m2c = eval_metrics(real_lbl, real_tech, real_act, np.argmax(s2_c[0], 1), s2_c[1], np.argmax(s2_c[2], 1), np.max(s2_c[0], 1))
    m2n = eval_metrics(real_lbl, real_tech, real_act, np.argmax(s2_n[0], 1), s2_n[1], np.argmax(s2_n[2], 1), np.max(s2_n[0], 1))
    m3c = eval_metrics(real_lbl, real_tech, real_act, np.argmax(s3_c[0], 1), s3_c[1], np.argmax(s3_c[2], 1), np.max(s3_c[0], 1))
    m3n = eval_metrics(real_lbl, real_tech, real_act, np.argmax(s3_n[0], 1), s3_n[1], np.argmax(s3_n[2], 1), np.max(s3_n[0], 1))
    print(f'=== {name} ===')
    print(f'  Split 1 CLEAN: A={m1c[0]:.4f} B={m1c[1]:.4f} C={m1c[2]:.4f} | per_cls={m1c[3]}')
    print(f'  Split 1 NOISY: A={m1n[0]:.4f} B={m1n[1]:.4f} C={m1n[2]:.4f} | per_cls={m1n[3]}')
    print(f'  Split 2 CLEAN: A={m2c[0]:.4f} B={m2c[1]:.4f} C={m2c[2]:.4f} | per_cls={m2c[3]}')
    print(f'  Split 2 NOISY: A={m2n[0]:.4f} B={m2n[1]:.4f} C={m2n[2]:.4f} | per_cls={m2n[3]}')
    print(f'  Split 3 CLEAN: A={m3c[0]:.4f} B={m3c[1]:.4f} C={m3c[2]:.4f} | per_cls={m3c[3]}')
    print(f'  Split 3 NOISY: A={m3n[0]:.4f} B={m3n[1]:.4f} C={m3n[2]:.4f} | per_cls={m3n[3]}')
    SUMMARY_RESULTS.append({
        'Model': name,
        'S1 Clean (A/B/C)': f'{m1c[0]:.3f}/{m1c[1]:.3f}/{m1c[2]:.3f}',
        'S1 Noisy (A/B/C)': f'{m1n[0]:.3f}/{m1n[1]:.3f}/{m1n[2]:.3f}',
        'S2 Real Clean (A/B/C)': f'{m2c[0]:.3f}/{m2c[1]:.3f}/{m2c[2]:.3f}',
        'S2 Real Noisy (A/B/C)': f'{m2n[0]:.3f}/{m2n[1]:.3f}/{m2n[2]:.3f}',
        'S3 Clean (A/B/C)': f'{m3c[0]:.3f}/{m3c[1]:.3f}/{m3c[2]:.3f}',
        'S3 Noisy (A/B/C)': f'{m3n[0]:.3f}/{m3n[1]:.3f}/{m3n[2]:.3f}',
        'Throughput': throughput,
    })

report_all_splits('TF-IDF + LR', (s1_pa_c, s1_pb_c, s1_pc_c), (s1_pa_n, s1_pb_n, s1_pc_n),
                  (s2_pa_c, s2_pb_c, s2_pc_c), (s2_pa_n, s2_pb_n, s2_pc_n),
                  (s3_pa_c, s3_pb_c, s3_pc_c), (s3_pa_n, s3_pb_n, s3_pc_n))


## 4. Experiment 2: Frozen Multilingual Embeddings + LogisticRegression

In [ ]:
@torch.no_grad()
def compute_embeddings(model_name, texts, prefix='', batch_size=128, max_len=256, cache_path=None):
    if cache_path and os.path.isfile(cache_path):
        return np.load(cache_path)
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    prefixed = [prefix + t for t in texts]
    embs = []
    for i in range(0, len(prefixed), batch_size):
        batch = prefixed[i:i+batch_size]
        enc = tok(batch, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(DEVICE)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda'), dtype=torch.float16):
            out = mdl(**enc)
            mask = enc['attention_mask'].unsqueeze(-1).expand(out.last_hidden_state.size()).float()
            pooled = torch.sum(out.last_hidden_state * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        embs.append(pooled.cpu().float().numpy())
    res = np.vstack(embs)
    if cache_path:
        np.save(cache_path, res)
    del mdl
    torch.cuda.empty_cache()
    return res

def run_frozen_emb_experiment(tag, model_name, prefix=''):
    pred_cache = os.path.join(WORK_DIR, f'preds_{tag}.npz')
    e_sc = compute_embeddings(model_name, synth_df['clean_text'].tolist(), prefix, cache_path=os.path.join(WORK_DIR, f'emb_{tag}_sc.npy'))
    e_sn = compute_embeddings(model_name, synth_df['noisy_text'].tolist(), prefix, cache_path=os.path.join(WORK_DIR, f'emb_{tag}_sn.npy'))
    e_rc = compute_embeddings(model_name, real_df['clean_text'].tolist(), prefix, cache_path=os.path.join(WORK_DIR, f'emb_{tag}_rc.npy'))
    e_rn = compute_embeddings(model_name, real_df['noisy_text'].tolist(), prefix, cache_path=os.path.join(WORK_DIR, f'emb_{tag}_rn.npy'))
    if os.path.isfile(pred_cache):
        d = np.load(pred_cache)
        s1_c = (d['s1_pa_c'], d['s1_pb_c'], d['s1_pc_c'])
        s1_n = (d['s1_pa_n'], d['s1_pb_n'], d['s1_pc_n'])
        s2_c = (d['s2_pa_c'], d['s2_pb_c'], d['s2_pc_c'])
        s2_n = (d['s2_pa_n'], d['s2_pb_n'], d['s2_pc_n'])
        s3_c = (d['s3_pa_c'], d['s3_pb_c'], d['s3_pc_c'])
        s3_n = (d['s3_pa_n'], d['s3_pb_n'], d['s3_pc_n'])
    else:
        def fit_pred_emb(X_tr, y_l, y_t, y_a, X_vc, X_vn, w=None):
            ca = LogisticRegression(C=5.0, class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_tr, y_l, sample_weight=w)
            cb = OneVsRestClassifier(LogisticRegression(C=5.0, class_weight='balanced', max_iter=1000, random_state=SEED)).fit(X_tr, y_t)
            cc = LogisticRegression(C=5.0, class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_tr, y_a, sample_weight=w)
            return ca.predict_proba(X_vc), cb.predict_proba(X_vc), cc.predict_proba(X_vc), ca.predict_proba(X_vn), cb.predict_proba(X_vn), cc.predict_proba(X_vn)
        n_s, n_r = len(synth_df), len(real_df)
        s1_pa_c, s1_pb_c, s1_pc_c = np.zeros((n_s, 6)), np.zeros((n_s, 6)), np.zeros((n_s, 6))
        s1_pa_n, s1_pb_n, s1_pc_n = np.zeros((n_s, 6)), np.zeros((n_s, 6)), np.zeros((n_s, 6))
        for tr_i, va_i in split1_folds:
            pac, pbc, pcc, pan, pbn, pcn = fit_pred_emb(e_sc[tr_i], synth_lbl[tr_i], synth_tech[tr_i], synth_act[tr_i], e_sc[va_i], e_sn[va_i])
            s1_pa_c[va_i], s1_pb_c[va_i], s1_pc_c[va_i] = pac, pbc, pcc
            s1_pa_n[va_i], s1_pb_n[va_i], s1_pc_n[va_i] = pan, pbn, pcn
        s2_pa_c, s2_pb_c, s2_pc_c, s2_pa_n, s2_pb_n, s2_pc_n = fit_pred_emb(
            e_sc[split2_train_idx], synth_lbl[split2_train_idx], synth_tech[split2_train_idx], synth_act[split2_train_idx], e_rc, e_rn
        )
        s3_pa_c, s3_pb_c, s3_pc_c = np.zeros((n_r, 6)), np.zeros((n_r, 6)), np.zeros((n_r, 6))
        s3_pa_n, s3_pb_n, s3_pc_n = np.zeros((n_r, 6)), np.zeros((n_r, 6)), np.zeros((n_r, 6))
        for tr_r, va_r in split3_folds:
            tr_pids = set(real_ids[tr_r])
            tr_para_idx = np.where(np.isin(para_parent_ids, list(tr_pids)))[0]
            synth_tr_idx = np.concatenate([s1_indices, tr_para_idx])
            X_tr = np.vstack([e_rc[tr_r], e_sc[synth_tr_idx]])
            tr_l = np.concatenate([real_lbl[tr_r], synth_lbl[synth_tr_idx]])
            tr_t = np.vstack([real_tech[tr_r], synth_tech[synth_tr_idx]])
            tr_a = np.concatenate([real_act[tr_r], synth_act[synth_tr_idx]])
            w = np.concatenate([np.full(len(tr_r), 5.0), np.ones(len(synth_tr_idx))])
            pac, pbc, pcc, pan, pbn, pcn = fit_pred_emb(X_tr, tr_l, tr_t, tr_a, e_rc[va_r], e_rn[va_r], w)
            s3_pa_c[va_r], s3_pb_c[va_r], s3_pc_c[va_r] = pac, pbc, pcc
            s3_pa_n[va_r], s3_pb_n[va_r], s3_pc_n[va_r] = pan, pbn, pcn
        np.savez(pred_cache, s1_pa_c=s1_pa_c, s1_pb_c=s1_pb_c, s1_pc_c=s1_pc_c, s1_pa_n=s1_pa_n, s1_pb_n=s1_pb_n, s1_pc_n=s1_pc_n,
                 s2_pa_c=s2_pa_c, s2_pb_c=s2_pb_c, s2_pc_c=s2_pc_c, s2_pa_n=s2_pa_n, s2_pb_n=s2_pb_n, s2_pc_n=s2_pc_n,
                 s3_pa_c=s3_pa_c, s3_pb_c=s3_pb_c, s3_pc_c=s3_pc_c, s3_pa_n=s3_pa_n, s3_pb_n=s3_pb_n, s3_pc_n=s3_pc_n)
        s1_c, s1_n = (s1_pa_c, s1_pb_c, s1_pc_c), (s1_pa_n, s1_pb_n, s1_pc_n)
        s2_c, s2_n = (s2_pa_c, s2_pb_c, s2_pc_c), (s2_pa_n, s2_pb_n, s2_pc_n)
        s3_c, s3_n = (s3_pa_c, s3_pb_c, s3_pc_c), (s3_pa_n, s3_pb_n, s3_pc_n)
    report_all_splits(f'Frozen {model_name}', s1_c, s1_n, s2_c, s2_n, s3_c, s3_n)
    return s2_c, s2_n

e5_s2_c, e5_s2_n = run_frozen_emb_experiment('e5', 'intfloat/multilingual-e5-base', prefix='query: ')
minilm_s2_c, minilm_s2_n = run_frozen_emb_experiment('minilm', 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', prefix='')


## 5. Experiment 3: Fine-Tune 3-Head `xlm-roberta-base` (Split 2 + Split 1 Fold 0)

In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, texts, lbls, techs, acts, weights, tokenizer, max_len=256):
        self.texts = texts
        self.lbls = lbls
        self.techs = techs
        self.acts = acts
        self.weights = weights
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = self.tok.encode(self.texts[idx], add_special_tokens=False)
        max_body = self.max_len - 2
        if len(ids) > max_body:
            # Head + tail truncation so delayed-reveal turns are preserved
            half = max_body // 2
            ids = ids[:half] + ids[-(max_body - half):]
        input_ids = [self.tok.cls_token_id] + ids + [self.tok.sep_token_id]
        return {
            'input_ids': input_ids,
            'lbl': int(self.lbls[idx]),
            'tech': self.techs[idx].astype(np.float32),
            'act': int(self.acts[idx]),
            'w': float(self.weights[idx]),
        }

def make_collate_fn(pad_id):
    def collate(batch):
        max_l = max(len(b['input_ids']) for b in batch)
        inp, mask = [], []
        for b in batch:
            l = len(b['input_ids'])
            inp.append(b['input_ids'] + [pad_id] * (max_l - l))
            mask.append([1] * l + [0] * (max_l - l))
        return {
            'input_ids': torch.tensor(inp, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'lbl': torch.tensor([b['lbl'] for b in batch], dtype=torch.long),
            'tech': torch.tensor(np.array([b['tech'] for b in batch]), dtype=torch.float32),
            'act': torch.tensor([b['act'] for b in batch], dtype=torch.long),
            'w': torch.tensor([b['w'] for b in batch], dtype=torch.float32),
        }
    return collate

class ThreeHeadEncoder(nn.Module):
    def __init__(self, model_name='xlm-roberta-base'):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        h = self.backbone.config.hidden_size
        self.drop = nn.Dropout(0.15)
        self.head_a = nn.Linear(h, 6)
        self.head_b = nn.Linear(h, 6)
        self.head_c = nn.Linear(h, 6)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(out.size()).float()
        pooled = torch.sum(out * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        x = self.drop(pooled)
        return self.head_a(x), self.head_b(x), self.head_c(x)

@torch.no_grad()
def predict_three_head(model, texts, tokenizer, batch_size=64, max_len=256):
    model.eval()
    ds = MultiTaskDataset(texts, np.zeros(len(texts)), np.zeros((len(texts), 6)), np.zeros(len(texts)), np.ones(len(texts)), tokenizer, max_len)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=make_collate_fn(tokenizer.pad_token_id))
    pa_list, pb_list, pc_list = [], [], []
    for batch in dl:
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda'), dtype=torch.float16):
            la, lb, lc = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        pa_list.append(torch.softmax(la.float(), dim=-1).cpu().numpy())
        pb_list.append(torch.sigmoid(lb.float()).cpu().numpy())
        pc_list.append(torch.softmax(lc.float(), dim=-1).cpu().numpy())
    return np.vstack(pa_list), np.vstack(pb_list), np.vstack(pc_list)

def train_three_head(run_name, tr_texts, tr_lbl, tr_tech, tr_act, tr_weights, ckpt_path, model_name='xlm-roberta-base', epochs=3, batch_size=32, lr=2e-5, max_len=256):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = ThreeHeadEncoder(model_name).to(DEVICE)
    if os.path.isfile(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'[{run_name}] Loaded cached checkpoint {ckpt_path}')
        return model, tok
    ds = MultiTaskDataset(tr_texts, tr_lbl, tr_tech, tr_act, tr_weights, tok, max_len)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=make_collate_fn(tok.pad_token_id))
    cw = compute_class_weight('balanced', classes=np.arange(6), y=tr_lbl)
    ce_a = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEVICE), reduction='none')
    bce_b = nn.BCEWithLogitsLoss(reduction='none')
    ce_c = nn.CrossEntropyLoss(reduction='none')
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.10 * total_steps), total_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    for ep in range(epochs):
        t_ep = time.time()
        model.train()
        tot_loss = 0.0
        for batch in dl:
            opt.zero_grad()
            inp = batch['input_ids'].to(DEVICE)
            msk = batch['attention_mask'].to(DEVICE)
            la_y, lb_y, lc_y = batch['lbl'].to(DEVICE), batch['tech'].to(DEVICE), batch['act'].to(DEVICE)
            sw = batch['w'].to(DEVICE)
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda'), dtype=torch.float16):
                la, lb, lc = model(inp, msk)
                loss_a = (ce_a(la, la_y) * sw).mean()
                loss_b = (bce_b(lb, lb_y).mean(dim=1) * sw).mean()
                loss_c = (ce_c(lc, lc_y) * sw).mean()
                loss = 0.5 * loss_a + 0.3 * loss_b + 0.2 * loss_c
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot_loss += loss.item()
        ep_time = time.time() - t_ep
        print(f'[{run_name}] Epoch {ep+1}/{epochs} | loss={tot_loss/len(dl):.4f} | time={ep_time:.1f}s')
        if ep == 0 and ep_time > 480 and model_name == 'xlm-roberta-base':
            print('WARNING: Epoch > 8 min; switching to MiniLM backbone recommended for speed.')
    torch.save(model.state_dict(), ckpt_path)
    return model, tok

BACKBONE = 'xlm-roberta-base'
# Split 2 training (synthetic_v2 only -> eval on 86 real rows)
xlmr_s2_ckpt = os.path.join(WORK_DIR, 'xlmr_split2.pt')
xlmr_s2_preds = os.path.join(WORK_DIR, 'preds_xlmr_s2.npz')
m_s2, tok_s2 = train_three_head(
    'XLM-R Split2', synth_df.loc[split2_train_idx, 'clean_text'].tolist(),
    synth_lbl[split2_train_idx], synth_tech[split2_train_idx], synth_act[split2_train_idx],
    np.ones(len(split2_train_idx)), xlmr_s2_ckpt, model_name=BACKBONE
)
if os.path.isfile(xlmr_s2_preds):
    d = np.load(xlmr_s2_preds)
    x_s2_c = (d['pa_c'], d['pb_c'], d['pc_c'])
    x_s2_n = (d['pa_n'], d['pb_n'], d['pc_n'])
else:
    pa_c, pb_c, pc_c = predict_three_head(m_s2, real_df['clean_text'].tolist(), tok_s2)
    pa_n, pb_n, pc_n = predict_three_head(m_s2, real_df['noisy_text'].tolist(), tok_s2)
    np.savez(xlmr_s2_preds, pa_c=pa_c, pb_c=pb_c, pc_c=pc_c, pa_n=pa_n, pb_n=pb_n, pc_n=pc_n)
    x_s2_c, x_s2_n = (pa_c, pb_c, pc_c), (pa_n, pb_n, pc_n)

# Split 1 Fold 0 only (to save time)
tr_f0, va_f0 = split1_folds[0]
xlmr_s1_ckpt = os.path.join(WORK_DIR, 'xlmr_split1_f0.pt')
xlmr_s1_preds = os.path.join(WORK_DIR, 'preds_xlmr_s1_f0.npz')
m_s1, tok_s1 = train_three_head(
    'XLM-R Split1-F0', synth_df.loc[tr_f0, 'clean_text'].tolist(),
    synth_lbl[tr_f0], synth_tech[tr_f0], synth_act[tr_f0],
    np.ones(len(tr_f0)), xlmr_s1_ckpt, model_name=BACKBONE
)
if os.path.isfile(xlmr_s1_preds):
    d1 = np.load(xlmr_s1_preds)
    x_s1_c = (d1['pa_c'], d1['pb_c'], d1['pc_c'])
    x_s1_n = (d1['pa_n'], d1['pb_n'], d1['pc_n'])
else:
    pa_c1, pb_c1, pc_c1 = predict_three_head(m_s1, synth_df.loc[va_f0, 'clean_text'].tolist(), tok_s1)
    pa_n1, pb_n1, pc_n1 = predict_three_head(m_s1, synth_df.loc[va_f0, 'noisy_text'].tolist(), tok_s1)
    np.savez(xlmr_s1_preds, pa_c=pa_c1, pb_c=pb_c1, pc_c=pc_c1, pa_n=pa_n1, pb_n=pb_n1, pc_n=pc_n1)
    x_s1_c, x_s1_n = (pa_c1, pb_c1, pc_c1), (pa_n1, pb_n1, pc_n1)

xm1c = eval_metrics(synth_lbl[va_f0], synth_tech[va_f0], synth_act[va_f0], np.argmax(x_s1_c[0], 1), x_s1_c[1], np.argmax(x_s1_c[2], 1), np.max(x_s1_c[0], 1))
xm1n = eval_metrics(synth_lbl[va_f0], synth_tech[va_f0], synth_act[va_f0], np.argmax(x_s1_n[0], 1), x_s1_n[1], np.argmax(x_s1_n[2], 1), np.max(x_s1_n[0], 1))
xm2c = eval_metrics(real_lbl, real_tech, real_act, np.argmax(x_s2_c[0], 1), x_s2_c[1], np.argmax(x_s2_c[2], 1), np.max(x_s2_c[0], 1))
xm2n = eval_metrics(real_lbl, real_tech, real_act, np.argmax(x_s2_n[0], 1), x_s2_n[1], np.argmax(x_s2_n[2], 1), np.max(x_s2_n[0], 1))
print(f'=== Fine-tuned {BACKBONE} ===')
print(f'  Split 1-F0 CLEAN: A={xm1c[0]:.4f} B={xm1c[1]:.4f} C={xm1c[2]:.4f} | per_cls={xm1c[3]}')
print(f'  Split 1-F0 NOISY: A={xm1n[0]:.4f} B={xm1n[1]:.4f} C={xm1n[2]:.4f} | per_cls={xm1n[3]}')
print(f'  Split 2    CLEAN: A={xm2c[0]:.4f} B={xm2c[1]:.4f} C={xm2c[2]:.4f} | per_cls={xm2c[3]}')
print(f'  Split 2    NOISY: A={xm2n[0]:.4f} B={xm2n[1]:.4f} C={xm2n[2]:.4f} | per_cls={xm2n[3]}')
SUMMARY_RESULTS.append({
    'Model': f'Fine-tuned {BACKBONE}',
    'S1 Clean (A/B/C)': f'{xm1c[0]:.3f}/{xm1c[1]:.3f}/{xm1c[2]:.3f} (F0)',
    'S1 Noisy (A/B/C)': f'{xm1n[0]:.3f}/{xm1n[1]:.3f}/{xm1n[2]:.3f} (F0)',
    'S2 Real Clean (A/B/C)': f'{xm2c[0]:.3f}/{xm2c[1]:.3f}/{xm2c[2]:.3f}',
    'S2 Real Noisy (A/B/C)': f'{xm2n[0]:.3f}/{xm2n[1]:.3f}/{xm2n[2]:.3f}',
    'S3 Clean (A/B/C)': '-',
    'S3 Noisy (A/B/C)': '-',
    'Throughput': 'See Benchmark',
})


## 6. Experiment 4: Inference Benchmark (10,000 Rows of `test.csv`, Batch 256, FP16, Length-Sorted Dynamic Padding)

In [ ]:
test_10k = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'), nrows=10000)
bench_texts = test_10k['text'].fillna('').astype(str).apply(preprocess_text).tolist()

# Length-sorted dynamic padding
order = np.argsort([len(t) for t in bench_texts])
sorted_texts = [bench_texts[i] for i in order]

m_s2.eval()
t_bench = time.time()
with torch.no_grad():
    for i in range(0, len(sorted_texts), 256):
        batch = sorted_texts[i:i+256]
        enc = tok_s2(batch, padding=True, truncation=True, max_length=256, return_tensors='pt').to(DEVICE)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda'), dtype=torch.float16):
            _ = m_s2(enc['input_ids'], enc['attention_mask'])
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
elapsed = time.time() - t_bench
rows_per_sec = len(sorted_texts) / elapsed
extrap_min = (500000 / rows_per_sec) / 60.0
print(f'Benchmark (10,000 rows, batch=256, fp16, length-sorted): {elapsed:.2f}s -> {rows_per_sec:.1f} rows/sec')
print(f'Extrapolated 500,000 rows time: {extrap_min:.2f} minutes')
if extrap_min > 60.0:
    print('WARNING: Extrapolated time > 60 minutes! Recommended fixes: reduce max_len to 128, use MiniLM-L12/L6 backbone, or export to ONNX Runtime FP16.')
SUMMARY_RESULTS[-1]['Throughput'] = f'{rows_per_sec:.1f} r/s ({extrap_min:.1f}m/500k)'


## 7. Final Summary Table & Best Model Error Analysis on Real Rows

In [ ]:
summary_df = pd.DataFrame(SUMMARY_RESULTS)
print('=== PHASE 3B SUMMARY TABLE ===')
print(summary_df.to_string(index=False))

# Compare Split 2 Clean Task A Macro F1 across all 4 models and print 15 errors for the best one
candidates = [
    ('TF-IDF + LR', s2_pa_c),
    ('Frozen E5-base', e5_s2_c[0]),
    ('Frozen MiniLM', minilm_s2_c[0]),
    (f'Fine-tuned {BACKBONE}', x_s2_c[0]),
]
best_name, best_pa = max(candidates, key=lambda x: f1_score(real_lbl, np.argmax(x[1], 1), average='macro'))
best_pred = np.argmax(best_pa, axis=1)
best_conf = np.max(best_pa, axis=1)
print(f'\n=== 15 Most Common / Highest-Confidence Errors for Best Model ({best_name}) on 86 Real Rows ===')
err_idx = np.where(best_pred != real_lbl)[0]
err_idx = err_idx[np.argsort(-best_conf[err_idx])][:15]
for rank, idx in enumerate(err_idx, 1):
    print(f'[{rank:02d}] true={real_lbl[idx]} -> pred={best_pred[idx]} (conf={best_conf[idx]:.3f}) | {repr(real_df.loc[idx, "text"][:110])}')


---
# Phase 4: Final Training, Blending, Calibration & Full 500k Test Inference
- **Noise-only rows:** ~300 programmatically generated random consonant/letter strings (`label=0`, `techniques='NONE'`, `action=0`).
- **Final training set:** All `synthetic_v2.csv` + 300 noise-only rows + 86 real rows (clean & noisy copies, weight `5x`).
- **Models:** Final 3-head `xlm-roberta-base` + Final `TF-IDF + LogisticRegression`.
- **Blending & Calibration:** Grid search `w in {0, 0.3, 0.5, 0.7, 1.0}` per task on held-out sets (preferring `0.5` unless another weight improves average F1 by `> 0.02`); Brier-optimal calibration among raw, temperature scaling, isotonic regression, and Platt scaling.


In [ ]:
from scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

def generate_noise_only_rows(n_rows=300, seed=SEED):
    rng_n = random.Random(seed + 999)
    consonants = 'bcdfghjklmnpqrstvwxyz'
    letters = 'abcdefghijklmnopqrstuvwxyz'
    emojis_n = ['📲', '⚠️', '🔥', '💰', '']
    puncts_n = ['', '.', '!', '?', '...']
    rows = []
    for _ in range(n_rows):
        n_tok = rng_n.randint(1, 3)
        toks = []
        for _ in range(n_tok):
            alphabet = consonants if rng_n.random() < 0.65 else letters
            tok = ''.join(rng_n.choice(alphabet) for _ in range(rng_n.randint(5, 13)))
            toks.append(tok)
        txt = ' '.join(toks)
        if rng_n.random() < 0.25:
            txt = txt + rng_n.choice(puncts_n)
        if rng_n.random() < 0.20:
            em = rng_n.choice(emojis_n)
            if em:
                txt = (em + ' ' + txt) if rng_n.random() < 0.5 else (txt + ' ' + em)
        rows.append(preprocess_text(txt))
    return rows

noise_only_texts = generate_noise_only_rows(300, seed=SEED)
n_noise = len(noise_only_texts)

final_texts = (
    synth_df['clean_text'].tolist()
    + noise_only_texts
    + real_df['clean_text'].tolist()
    + real_df['noisy_text'].tolist()
)
final_lbl = np.concatenate([
    synth_lbl,
    np.zeros(n_noise, dtype=int),
    real_lbl,
    real_lbl,
])
final_tech = np.vstack([
    synth_tech,
    np.zeros((n_noise, 6), dtype=int),
    real_tech,
    real_tech,
])
final_act = np.concatenate([
    synth_act,
    np.zeros(n_noise, dtype=int),
    real_act,
    real_act,
])
final_weights = np.concatenate([
    np.ones(len(synth_df), dtype=np.float32),
    np.ones(n_noise, dtype=np.float32),
    np.full(len(real_df), 5.0, dtype=np.float32),
    np.full(len(real_df), 5.0, dtype=np.float32),
])
print(f'Final training set: {len(final_texts):,} rows (synth_v2={len(synth_df)}, noise_only={n_noise}, real_clean+noisy={2*len(real_df)} @ 5x weight)')


In [ ]:
import pickle

# (a) Final XLM-R three-head model
xlmr_final_ckpt = os.path.join(WORK_DIR, 'xlmr_final.pt')
m_final, tok_final = train_three_head(
    'XLM-R Final', final_texts, final_lbl, final_tech, final_act,
    final_weights, xlmr_final_ckpt, model_name=BACKBONE, epochs=3, batch_size=32, lr=2e-5, max_len=256
)

# (b) Final TF-IDF char+word + LogisticRegression models
tfidf_final_cache = os.path.join(WORK_DIR, 'tfidf_final_models.pkl')
if os.path.isfile(tfidf_final_cache):
    with open(tfidf_final_cache, 'rb') as f:
        tf_bundle = pickle.load(f)
    fin_cv, fin_wv = tf_bundle['cv'], tf_bundle['wv']
    fin_ca, fin_cb, fin_cc = tf_bundle['ca'], tf_bundle['cb'], tf_bundle['cc']
    print('Loaded cached final TF-IDF models from', tfidf_final_cache)
else:
    t_tf = time.time()
    fin_cv = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True, dtype=np.float32)
    fin_wv = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32)
    X_fin = hstack([fin_cv.fit_transform(final_texts), fin_wv.fit_transform(final_texts)]).tocsr()
    fin_ca = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_fin, final_lbl, sample_weight=final_weights)
    fin_cb = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)).fit(X_fin, final_tech)
    fin_cc = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_fin, final_act, sample_weight=final_weights)
    with open(tfidf_final_cache, 'wb') as f:
        pickle.dump({'cv': fin_cv, 'wv': fin_wv, 'ca': fin_ca, 'cb': fin_cb, 'cc': fin_cc}, f)
    print(f'Trained & cached final TF-IDF models in {time.time()-t_tf:.1f}s')

# Training alphabetic vocabulary for the noise-only OOV rule
TRAIN_ALPHA_VOCAB = set()
for txt in final_texts:
    for tok in txt.split():
        if tok not in ('[URL]', '[NUM]', '|||') and any(c.isalpha() for c in tok):
            TRAIN_ALPHA_VOCAB.add(tok.strip('.,!?:;"\'()[]{}'))
print('Training alphabetic vocabulary size:', len(TRAIN_ALPHA_VOCAB))


In [ ]:
# Load held-out predictions from Phase 3B
d_tf = np.load(os.path.join(WORK_DIR, 'preds_tfidf.npz'))
d_x2 = np.load(os.path.join(WORK_DIR, 'preds_xlmr_s2.npz'))
d_x1 = np.load(os.path.join(WORK_DIR, 'preds_xlmr_s1_f0.npz'))
_, va_f0 = split1_folds[0]

weights_grid = [0.0, 0.3, 0.5, 0.7, 1.0]
blend_rows = []
best_w = {}

for task_name, key_xlmr, key_tf_s2, key_tf_s1, is_multilabel in [
    ('A', 'pa', 's2_pa', 's1_pa', False),
    ('B', 'pb', 's2_pb', 's1_pb', True),
    ('C', 'pc', 's2_pc', 's1_pc', False),
]:
    scores_by_w = {}
    for w in weights_grid:
        p_s2c = w * d_x2[f'{key_xlmr}_c'] + (1.0 - w) * d_tf[f'{key_tf_s2}_c']
        p_s2n = w * d_x2[f'{key_xlmr}_n'] + (1.0 - w) * d_tf[f'{key_tf_s2}_n']
        p_s1c = w * d_x1[f'{key_xlmr}_c'] + (1.0 - w) * d_tf[f'{key_tf_s1}_c'][va_f0]
        p_s1n = w * d_x1[f'{key_xlmr}_n'] + (1.0 - w) * d_tf[f'{key_tf_s1}_n'][va_f0]

        if task_name == 'A':
            f_s2c = f1_score(real_lbl, np.argmax(p_s2c, 1), average='macro')
            f_s2n = f1_score(real_lbl, np.argmax(p_s2n, 1), average='macro')
            f_s1c = f1_score(synth_lbl[va_f0], np.argmax(p_s1c, 1), average='macro')
            f_s1n = f1_score(synth_lbl[va_f0], np.argmax(p_s1n, 1), average='macro')
        elif task_name == 'B':
            f_s2c = float(np.mean([f1_score(real_tech[:, j], (p_s2c[:, j] > 0.5).astype(int), zero_division=0) for j in range(6)]))
            f_s2n = float(np.mean([f1_score(real_tech[:, j], (p_s2n[:, j] > 0.5).astype(int), zero_division=0) for j in range(6)]))
            f_s1c = float(np.mean([f1_score(synth_tech[va_f0, j], (p_s1c[:, j] > 0.5).astype(int), zero_division=0) for j in range(6)]))
            f_s1n = float(np.mean([f1_score(synth_tech[va_f0, j], (p_s1n[:, j] > 0.5).astype(int), zero_division=0) for j in range(6)]))
        else:
            f_s2c = f1_score(real_act, np.argmax(p_s2c, 1), average='macro')
            f_s2n = f1_score(real_act, np.argmax(p_s2n, 1), average='macro')
            f_s1c = f1_score(synth_act[va_f0], np.argmax(p_s1c, 1), average='macro')
            f_s1n = f1_score(synth_act[va_f0], np.argmax(p_s1n, 1), average='macro')

        avg_score = (f_s2c + f_s2n + f_s1c + f_s1n) / 4.0
        scores_by_w[w] = avg_score
        blend_rows.append({
            'Task': task_name, 'w_XLM_R': w,
            'S2_Clean': round(f_s2c, 4), 'S2_Noisy': round(f_s2n, 4),
            'S1F0_Clean': round(f_s1c, 4), 'S1F0_Noisy': round(f_s1n, 4),
            'Avg_F1': round(avg_score, 4),
        })
    # Prefer 0.5 unless another weight is clearly better (> 0.02)
    max_w = max(scores_by_w, key=scores_by_w.get)
    chosen_w = max_w if (scores_by_w[max_w] - scores_by_w[0.5] > 0.02) else 0.5
    best_w[task_name] = chosen_w

blend_df = pd.DataFrame(blend_rows)
print('=== BLEND WEIGHT MACRO F1 TABLE ===')
print(blend_df.to_string(index=False))
print('Selected blend weights (w * XLM-R + (1-w) * TF-IDF):', best_w)


In [ ]:
wA, wB, wC = best_w['A'], best_w['B'], best_w['C']

# Blended held-out probabilities (Split 2 clean, Split 2 noisy, Split 1 F0 clean & noisy)
ho_pa = np.vstack([
    wA * d_x2['pa_c'] + (1.0 - wA) * d_tf['s2_pa_c'],
    wA * d_x2['pa_n'] + (1.0 - wA) * d_tf['s2_pa_n'],
    wA * d_x1['pa_c'] + (1.0 - wA) * d_tf['s1_pa_c'][va_f0],
    wA * d_x1['pa_n'] + (1.0 - wA) * d_tf['s1_pa_n'][va_f0],
])
ho_y_a = np.concatenate([real_lbl, real_lbl, synth_lbl[va_f0], synth_lbl[va_f0]])
ho_texts = (
    real_df['text'].tolist()
    + real_df['noisy_text'].tolist()
    + synth_df.loc[va_f0, 'text'].tolist()
    + synth_df.loc[va_f0, 'noisy_text'].tolist()
)

ho_pred_a = np.argmax(ho_pa, axis=1)
ho_correct = (ho_pred_a == ho_y_a).astype(int)
ho_raw_max = np.max(ho_pa, axis=1)

# 1. Raw max probability Brier
brier_raw = brier_score_loss(ho_correct, ho_raw_max)

# 2. Temperature scaling on blended probabilities
log_ho_pa = np.log(np.clip(ho_pa, 1e-12, 1.0))
def temp_nll(T):
    scaled = log_ho_pa / T
    scaled -= np.max(scaled, axis=1, keepdims=True)
    ex = np.exp(scaled)
    probs = ex / np.sum(ex, axis=1, keepdims=True)
    p_true = np.clip(probs[np.arange(len(ho_y_a)), ho_y_a], 1e-12, 1.0)
    return -np.mean(np.log(p_true))
opt_T = float(minimize_scalar(temp_nll, bounds=(0.2, 5.0), method='bounded').x)
def apply_temp(probs, T):
    s = np.log(np.clip(probs, 1e-12, 1.0)) / T
    s -= np.max(s, axis=1, keepdims=True)
    ex = np.exp(s)
    p = ex / np.sum(ex, axis=1, keepdims=True)
    return np.max(p, axis=1)
ho_temp_max = apply_temp(ho_pa, opt_T)
brier_temp = brier_score_loss(ho_correct, ho_temp_max)

# 3. Isotonic Regression on max_prob -> correct
iso_cal = IsotonicRegression(y_min=0.05, y_max=0.98, out_of_bounds='clip').fit(ho_raw_max, ho_correct)
ho_iso = iso_cal.predict(ho_raw_max)
brier_iso = brier_score_loss(ho_correct, ho_iso)

# 4. Platt Scaling (LogisticRegression on max_prob -> correct)
platt_cal = LogisticRegression(C=1.0, max_iter=500, random_state=SEED).fit(ho_raw_max.reshape(-1, 1), ho_correct)
ho_platt = platt_cal.predict_proba(ho_raw_max.reshape(-1, 1))[:, 1]
brier_platt = brier_score_loss(ho_correct, ho_platt)

cal_scores = {'raw': brier_raw, 'temperature': brier_temp, 'isotonic': brier_iso, 'platt': brier_platt}
best_cal_method = min(cal_scores, key=cal_scores.get)
print('=== HELD-OUT BRIER SCORES ===')
for k, v in cal_scores.items():
    print(f'  {k:12s}: {v:.5f}')
print(f'Selected calibrator: {best_cal_method} (opt_T={opt_T:.3f})')

def calibrate_conf(probs_a):
    raw_m = np.max(probs_a, axis=1)
    if best_cal_method == 'raw':
        c = raw_m
    elif best_cal_method == 'temperature':
        c = apply_temp(probs_a, opt_T)
    elif best_cal_method == 'isotonic':
        c = iso_cal.predict(raw_m)
    else:
        c = platt_cal.predict_proba(raw_m.reshape(-1, 1))[:, 1]
    return np.round(np.clip(c, 0.05, 0.98), 4)

ho_cal_conf = calibrate_conf(ho_pa)
print(f'Held-out Accuracy: {ho_correct.mean():.4f} | Mean Calibrated Confidence: {ho_cal_conf.mean():.4f}')

# 15 held-out errors with highest confidence
err_mask = np.where(ho_pred_a != ho_y_a)[0]
top15_err = err_mask[np.argsort(-ho_cal_conf[err_mask])][:15]
print('\n=== 15 HELD-OUT ERRORS WITH HIGHEST CONFIDENCE ===')
for rank, idx in enumerate(top15_err, 1):
    print(f'[{rank:02d}] true={ho_y_a[idx]} -> pred={ho_pred_a[idx]} (conf={ho_cal_conf[idx]:.4f}) | {repr(str(ho_texts[idx])[:110])}')


In [ ]:
test_df = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))
N_TEST = len(test_df)
print(f'Running full inference on test.csv ({N_TEST:,} rows)...')

CHUNK_SIZE = 50000
m_final.eval()

# 1. XLM-R batched inference in 50k chunks (length-sorted dynamic padding, restored order, cached)
t_xlmr_all = time.time()
xlmr_pa_chunks, xlmr_pb_chunks, xlmr_pc_chunks = [], [], []
for start in range(0, N_TEST, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, N_TEST)
    c_idx = start // CHUNK_SIZE
    c_file = os.path.join(WORK_DIR, f'test_xlmr_chunk_{c_idx}.npz')
    if os.path.isfile(c_file):
        d = np.load(c_file)
        xlmr_pa_chunks.append(d['pa']); xlmr_pb_chunks.append(d['pb']); xlmr_pc_chunks.append(d['pc'])
        print(f'[XLM-R] Loaded cached chunk {c_idx+1} ({end:,}/{N_TEST:,})')
        continue
    t_c = time.time()
    raw_slice = test_df['text'].iloc[start:end].fillna('').astype(str).apply(preprocess_text).tolist()
    order = np.argsort([len(t) for t in raw_slice])
    sorted_slice = [raw_slice[i] for i in order]
    pa_s, pb_s, pc_s = predict_three_head(m_final, sorted_slice, tok_final, batch_size=256, max_len=256)
    inv_order = np.argsort(order)
    pa_c, pb_c, pc_c = pa_s[inv_order], pb_s[inv_order], pc_s[inv_order]
    np.savez(c_file, pa=pa_c, pb=pb_c, pc=pc_c)
    xlmr_pa_chunks.append(pa_c); xlmr_pb_chunks.append(pb_c); xlmr_pc_chunks.append(pc_c)
    dt = time.time() - t_c
    rps = len(raw_slice) / dt
    eta_min = ((N_TEST - end) / rps) / 60.0
    print(f'[XLM-R] Chunk {c_idx+1}: {end:,}/{N_TEST:,} | {rps:.1f} rows/s | ETA: {eta_min:.1f} min')

test_xlmr_pa = np.vstack(xlmr_pa_chunks)
test_xlmr_pb = np.vstack(xlmr_pb_chunks)
test_xlmr_pc = np.vstack(xlmr_pc_chunks)
print(f'XLM-R total inference time: {(time.time()-t_xlmr_all)/60:.2f} min')

# 2. TF-IDF batched inference in 50k chunks (cached)
tf_test_cache = os.path.join(WORK_DIR, 'test_tfidf_probs.npz')
if os.path.isfile(tf_test_cache):
    dtf = np.load(tf_test_cache)
    test_tf_pa, test_tf_pb, test_tf_pc = dtf['pa'], dtf['pb'], dtf['pc']
    print('Loaded cached TF-IDF test probabilities from', tf_test_cache)
else:
    t_tf_all = time.time()
    tf_pa_l, tf_pb_l, tf_pc_l = [], [], []
    for start in range(0, N_TEST, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, N_TEST)
        t_c = time.time()
        clean_slice = test_df['text'].iloc[start:end].fillna('').astype(str).apply(preprocess_text).tolist()
        X_t = hstack([fin_cv.transform(clean_slice), fin_wv.transform(clean_slice)]).tocsr()
        tf_pa_l.append(fin_ca.predict_proba(X_t).astype(np.float32))
        tf_pb_l.append(fin_cb.predict_proba(X_t).astype(np.float32))
        tf_pc_l.append(fin_cc.predict_proba(X_t).astype(np.float32))
        dt = time.time() - t_c
        rps = len(clean_slice) / dt
        eta_min = ((N_TEST - end) / rps) / 60.0
        print(f'[TF-IDF] Chunk {start//CHUNK_SIZE + 1}: {end:,}/{N_TEST:,} | {rps:.1f} rows/s | ETA: {eta_min:.2f} min')
    test_tf_pa = np.vstack(tf_pa_l)
    test_tf_pb = np.vstack(tf_pb_l)
    test_tf_pc = np.vstack(tf_pc_l)
    np.savez(tf_test_cache, pa=test_tf_pa, pb=test_tf_pb, pc=test_tf_pc)
    print(f'TF-IDF total inference time: {time.time()-t_tf_all:.1f}s')


In [ ]:
blend_pa = wA * test_xlmr_pa + (1.0 - wA) * test_tf_pa
blend_pb = wB * test_xlmr_pb + (1.0 - wB) * test_tf_pb
blend_pc = wC * test_xlmr_pc + (1.0 - wC) * test_tf_pc

pred_labels = np.argmax(blend_pa, axis=1)
pred_actions = np.argmax(blend_pc, axis=1)
pred_confs = calibrate_conf(blend_pa)

def is_oov_noise_only(clean_txt: str) -> bool:
    if '[URL]' in clean_txt or '[NUM]' in clean_txt:
        return False
    alpha_toks = [
        tok.strip('.,!?:;"\'()[]{}')
        for tok in clean_txt.split()
        if tok != '|||' and any(c.isalpha() for c in tok)
    ]
    if not alpha_toks:
        return False
    return all(t not in TRAIN_ALPHA_VOCAB for t in alpha_toks)

clean_all_test = test_df['text'].fillna('').astype(str).apply(preprocess_text).tolist()
pred_techs = []
noise_rule_hits = 0

for i in range(N_TEST):
    if is_oov_noise_only(clean_all_test[i]):
        pred_labels[i] = 0
        pred_techs.append('NONE')
        pred_actions[i] = 0
        pred_confs[i] = 0.5
        noise_rule_hits += 1
        continue
    lbl = int(pred_labels[i])
    if lbl in (0, 5):
        pred_techs.append('NONE')
        pred_actions[i] = 0
    else:
        active = [TECH_CODES[j] for j in range(6) if blend_pb[i, j] > 0.5]
        if not active:
            active = [TECH_CODES[int(np.argmax(blend_pb[i]))]]
        pred_techs.append('|'.join(sorted(active)))

print(f'Noise-only OOV rule triggered on {noise_rule_hits:,} test rows.')

sub_final_path = os.path.join(WORK_DIR, 'submission_final.csv')
sub_final = pd.DataFrame({
    'id': test_df['id'].values.astype(int),
    'label': pred_labels.astype(int),
    'techniques': pred_techs,
    'action': pred_actions.astype(int),
    'confidence': pred_confs,
})
sub_final.to_csv(sub_final_path, index=False)
print(f'Wrote {sub_final_path} ({len(sub_final):,} rows)')

# Inline validator
ref_ids = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'), usecols=['id'])['id'].values
sub_chk = pd.read_csv(sub_final_path, keep_default_na=False)
VALID_CODES = {'T1', 'T2', 'T3', 'T4', 'T5', 'T6'}
def _chk_tech(v):
    if not isinstance(v, str) or v != v.strip() or not v or ' ' in v:
        return False
    if v == 'NONE':
        return True
    p = v.split('|')
    return 'NONE' not in p and all(x in VALID_CODES for x in p) and len(p) == len(set(p)) and p == sorted(p)

assert list(sub_chk.columns) == ['id', 'label', 'techniques', 'action', 'confidence'], 'Header mismatch'
assert len(sub_chk) == N_TEST == len(ref_ids), 'Row count mismatch'
assert (sub_chk['id'].astype(int).values == test_df['id'].values).all(), 'ID order mismatch vs test.csv'
assert not sub_chk['id'].duplicated().any(), 'Duplicate IDs'
for col in sub_chk.columns:
    assert (sub_chk[col].astype(str).str.strip() != '').all(), f'Blank values in {col}'
assert sub_chk['label'].astype(int).isin([0,1,2,3,4,5]).all(), 'Invalid label'
assert sub_chk['action'].astype(int).isin([0,1,2,3,4,5]).all(), 'Invalid action'
assert sub_chk['techniques'].apply(_chk_tech).all(), 'Invalid techniques'
c_chk = pd.to_numeric(sub_chk['confidence'], errors='raise')
assert not c_chk.isna().any() and ((c_chk >= 0.0) & (c_chk <= 1.0)).all(), 'Invalid confidence'
print('PASS: submission_final.csv passed all validator checks!')

# Label distribution vs Phase 2
phase2_pct = {0: 40.40, 1: 18.54, 2: 8.36, 3: 8.35, 4: 11.91, 5: 12.45}
print('\nPredicted Label Distribution (Phase 4 Final vs Phase 2):')
vc = sub_chk['label'].astype(int).value_counts().sort_index()
for k in range(6):
    cnt = int(vc.get(k, 0))
    pct = cnt / N_TEST * 100
    print(f'  Label {k}: {cnt:7,d} ({pct:5.2f}%)  [Phase 2: {phase2_pct[k]:5.2f}%]')

print('\nFiles in /kaggle/working:')
for fn in sorted(os.listdir(WORK_DIR)):
    print(' ', fn)


---
# Phase 5: Multi-Seed XLM-R Ensemble, Task B Threshold Tuning & `submission_v2.csv`
- **Second XLM-R (`seed=7`):** Trained on Split 2 (for 2-seed held-out check) and on the full final training set (`xlmr_final_seed7.pt`).
- **Ensemble:** `XLM-R_mean = 0.5 * (XLM-R_seed42 + XLM-R_seed7)`, then blended with TF-IDF using Phase 4 weights `(wA, wB, wC)`.
- **Task B Thresholds:** Tuned per technique (`0.30..0.70` step `0.05`) and shrunk 50% toward `0.5`; kept only if average Task B Macro F1 improves by `> 0.01` on BOTH Split 2 and Split 1 Fold 0.
- **Output:** `/kaggle/working/submission_v2.csv` (`submission_final.csv` is untouched).


In [ ]:
import gc

# Free the models from earlier experiments (their predictions are already cached)
for name in ['m_s2', 'm_s1', 'm_s2_s7', 'm_final_s7']:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print('GPU allocated (GB):', round(torch.cuda.memory_allocated() / 1e9, 2))
print('Other models still in memory:', [k for k, v in globals().items() if isinstance(v, nn.Module)])

seed_everything(7)

# 1A. Split 2 (seed=7) to verify 2-seed averaging on 86 real rows
xlmr_s2_s7_ckpt = os.path.join(WORK_DIR, 'xlmr_split2_seed7.pt')
xlmr_s2_s7_preds = os.path.join(WORK_DIR, 'preds_xlmr_s2_seed7.npz')
m_s2_s7, tok_s2_s7 = train_three_head(
    'XLM-R Split2 (seed=7)', synth_df.loc[split2_train_idx, 'clean_text'].tolist(),
    synth_lbl[split2_train_idx], synth_tech[split2_train_idx], synth_act[split2_train_idx],
    np.ones(len(split2_train_idx)), xlmr_s2_s7_ckpt, model_name=BACKBONE
)
if os.path.isfile(xlmr_s2_s7_preds):
    d_s7 = np.load(xlmr_s2_s7_preds)
    s2_s7_c = (d_s7['pa_c'], d_s7['pb_c'], d_s7['pc_c'])
    s2_s7_n = (d_s7['pa_n'], d_s7['pb_n'], d_s7['pc_n'])
else:
    pa_c7, pb_c7, pc_c7 = predict_three_head(m_s2_s7, real_df['clean_text'].tolist(), tok_s2_s7)
    pa_n7, pb_n7, pc_n7 = predict_three_head(m_s2_s7, real_df['noisy_text'].tolist(), tok_s2_s7)
    np.savez(xlmr_s2_s7_preds, pa_c=pa_c7, pb_c=pb_c7, pc_c=pc_c7, pa_n=pa_n7, pb_n=pb_n7, pc_n=pc_n7)
    s2_s7_c, s2_s7_n = (pa_c7, pb_c7, pc_c7), (pa_n7, pb_n7, pc_n7)

s2_mean_c = tuple(0.5 * (d_x2[f'{k}_c'] + s2_s7_c[idx]) for idx, k in enumerate(['pa', 'pb', 'pc']))
s2_mean_n = tuple(0.5 * (d_x2[f'{k}_n'] + s2_s7_n[idx]) for idx, k in enumerate(['pa', 'pb', 'pc']))

print('=== SPLIT 2 (86 REAL ROWS) COMPARISON: SEED 42 vs SEED 7 vs MEAN ===')
for tag_m, tup_c, tup_n in [
    ('Seed 42 alone', (d_x2['pa_c'], d_x2['pb_c'], d_x2['pc_c']), (d_x2['pa_n'], d_x2['pb_n'], d_x2['pc_n'])),
    ('Seed 7  alone', s2_s7_c, s2_s7_n),
    ('Mean (42 + 7)', s2_mean_c, s2_mean_n),
]:
    mc = eval_metrics(real_lbl, real_tech, real_act, np.argmax(tup_c[0], 1), tup_c[1], np.argmax(tup_c[2], 1), np.max(tup_c[0], 1))
    mn = eval_metrics(real_lbl, real_tech, real_act, np.argmax(tup_n[0], 1), tup_n[1], np.argmax(tup_n[2], 1), np.max(tup_n[0], 1))
    print(f'  {tag_m:14s} | CLEAN: A={mc[0]:.4f} B={mc[1]:.4f} C={mc[2]:.4f} | NOISY: A={mn[0]:.4f} B={mn[1]:.4f} C={mn[2]:.4f}')

# 1B. Final XLM-R (seed=7) on full final training set
xlmr_final_s7_ckpt = os.path.join(WORK_DIR, 'xlmr_final_seed7.pt')
m_final_s7, tok_final_s7 = train_three_head(
    'XLM-R Final (seed=7)', final_texts, final_lbl, final_tech, final_act,
    final_weights, xlmr_final_s7_ckpt, model_name=BACKBONE, epochs=3, batch_size=32, lr=2e-5, max_len=256
)
seed_everything(SEED)


In [ ]:
# Task B blended held-out probabilities (using 2-seed XLM-R mean on Split 2, seed 42 on Split 1 F0)
b_s2_c = wB * s2_mean_c[1] + (1.0 - wB) * d_tf['s2_pb_c']
b_s2_n = wB * s2_mean_n[1] + (1.0 - wB) * d_tf['s2_pb_n']
b_s1_c = wB * d_x1['pb_c'] + (1.0 - wB) * d_tf['s1_pb_c'][va_f0]
b_s1_n = wB * d_x1['pb_n'] + (1.0 - wB) * d_tf['s1_pb_n'][va_f0]

b_s2_all = np.vstack([b_s2_c, b_s2_n])
y_s2_b_all = np.vstack([real_tech, real_tech])
b_s1_all = np.vstack([b_s1_c, b_s1_n])
y_s1_b_all = np.vstack([synth_tech[va_f0], synth_tech[va_f0]])
b_ho_all = np.vstack([b_s2_all, b_s1_all])
y_ho_b_all = np.vstack([y_s2_b_all, y_s1_b_all])

grid_t = np.round(np.arange(0.30, 0.71, 0.05), 2)
shrunk_thresh = np.full(6, 0.5, dtype=float)
print('=== TASK B PER-TECHNIQUE THRESHOLD TUNING ===')
for j, code in enumerate(TECH_CODES):
    f1_before = f1_score(y_ho_b_all[:, j], (b_ho_all[:, j] >= 0.5).astype(int), zero_division=0)
    best_t, best_f1 = 0.5, -1.0
    for t in grid_t:
        sc = f1_score(y_ho_b_all[:, j], (b_ho_all[:, j] >= t).astype(int), zero_division=0)
        if sc > best_f1:
            best_f1, best_t = sc, float(t)
    t_shrunk = 0.5 + 0.5 * (best_t - 0.5)
    shrunk_thresh[j] = t_shrunk
    f1_after = f1_score(y_ho_b_all[:, j], (b_ho_all[:, j] >= t_shrunk).astype(int), zero_division=0)
    print(f'  {code}: tuned={best_t:.2f} -> shrunk={t_shrunk:.3f} | F1_before={f1_before:.4f} -> F1_after={f1_after:.4f}')

s2_b_before = float(np.mean([f1_score(y_s2_b_all[:, j], (b_s2_all[:, j] >= 0.5).astype(int), zero_division=0) for j in range(6)]))
s2_b_after = float(np.mean([f1_score(y_s2_b_all[:, j], (b_s2_all[:, j] >= shrunk_thresh[j]).astype(int), zero_division=0) for j in range(6)]))
s1_b_before = float(np.mean([f1_score(y_s1_b_all[:, j], (b_s1_all[:, j] >= 0.5).astype(int), zero_division=0) for j in range(6)]))
s1_b_after = float(np.mean([f1_score(y_s1_b_all[:, j], (b_s1_all[:, j] >= shrunk_thresh[j]).astype(int), zero_division=0) for j in range(6)]))
print(f'Split 2 Task B Macro F1: {s2_b_before:.4f} -> {s2_b_after:.4f} (diff={s2_b_after - s2_b_before:+.4f})')
print(f'Split 1 Task B Macro F1: {s1_b_before:.4f} -> {s1_b_after:.4f} (diff={s1_b_after - s1_b_before:+.4f})')

if (s2_b_after - s2_b_before > 0.01) and (s1_b_after - s1_b_before > 0.01):
    FINAL_TECH_THRESH = shrunk_thresh
    print('KEEPING tuned+shrunk thresholds:', np.round(FINAL_TECH_THRESH, 3))
else:
    FINAL_TECH_THRESH = np.full(6, 0.5, dtype=float)
    print('REVERTING to 0.5 thresholds (improvement <= 0.01 on one or both held-out sets).')

# Re-calibrate confidence on 2-seed blended held-out Task A probabilities
ho_pa_v2 = np.vstack([
    wA * s2_mean_c[0] + (1.0 - wA) * d_tf['s2_pa_c'],
    wA * s2_mean_n[0] + (1.0 - wA) * d_tf['s2_pa_n'],
    wA * d_x1['pa_c'] + (1.0 - wA) * d_tf['s1_pa_c'][va_f0],
    wA * d_x1['pa_n'] + (1.0 - wA) * d_tf['s1_pa_n'][va_f0],
])
ho_pred_v2 = np.argmax(ho_pa_v2, axis=1)
ho_corr_v2 = (ho_pred_v2 == ho_y_a).astype(int)
ho_raw_v2 = np.max(ho_pa_v2, axis=1)

brier_raw_v2 = brier_score_loss(ho_corr_v2, ho_raw_v2)
log_ho_v2 = np.log(np.clip(ho_pa_v2, 1e-12, 1.0))
def temp_nll_v2(T):
    sc = log_ho_v2 / T
    sc -= np.max(sc, axis=1, keepdims=True)
    ex = np.exp(sc)
    pr = ex / np.sum(ex, axis=1, keepdims=True)
    return -np.mean(np.log(np.clip(pr[np.arange(len(ho_y_a)), ho_y_a], 1e-12, 1.0)))
opt_T_v2 = float(minimize_scalar(temp_nll_v2, bounds=(0.2, 5.0), method='bounded').x)
ho_temp_v2 = apply_temp(ho_pa_v2, opt_T_v2)
brier_temp_v2 = brier_score_loss(ho_corr_v2, ho_temp_v2)

iso_v2 = IsotonicRegression(y_min=0.05, y_max=0.98, out_of_bounds='clip').fit(ho_raw_v2, ho_corr_v2)
brier_iso_v2 = brier_score_loss(ho_corr_v2, iso_v2.predict(ho_raw_v2))

platt_v2 = LogisticRegression(C=1.0, max_iter=500, random_state=SEED).fit(ho_raw_v2.reshape(-1, 1), ho_corr_v2)
brier_platt_v2 = brier_score_loss(ho_corr_v2, platt_v2.predict_proba(ho_raw_v2.reshape(-1, 1))[:, 1])

cal_v2_scores = {'raw': brier_raw_v2, 'temperature': brier_temp_v2, 'isotonic': brier_iso_v2, 'platt': brier_platt_v2}
best_cal_v2 = min(cal_v2_scores, key=cal_v2_scores.get)
print('\n=== PHASE 5 HELD-OUT BRIER SCORES ===')
for k, v in cal_v2_scores.items():
    print(f'  {k:12s}: {v:.5f}')
print(f'Selected Phase 5 calibrator: {best_cal_v2} (opt_T={opt_T_v2:.3f})')

def calibrate_conf_v2(probs_a):
    raw_m = np.max(probs_a, axis=1)
    if best_cal_v2 == 'raw':
        c = raw_m
    elif best_cal_v2 == 'temperature':
        c = apply_temp(probs_a, opt_T_v2)
    elif best_cal_v2 == 'isotonic':
        c = iso_v2.predict(raw_m)
    else:
        c = platt_v2.predict_proba(raw_m.reshape(-1, 1))[:, 1]
    return np.round(np.clip(c, 0.05, 0.98), 4)


In [ ]:
m_final_s7.eval()
t_s7_all = time.time()
s7_pa_chunks, s7_pb_chunks, s7_pc_chunks = [], [], []
for start in range(0, N_TEST, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, N_TEST)
    c_idx = start // CHUNK_SIZE
    c_file = os.path.join(WORK_DIR, f'test_xlmr_s7_chunk_{c_idx}.npz')
    if os.path.isfile(c_file):
        d = np.load(c_file)
        s7_pa_chunks.append(d['pa']); s7_pb_chunks.append(d['pb']); s7_pc_chunks.append(d['pc'])
        print(f'[XLM-R seed=7] Loaded cached chunk {c_idx+1} ({end:,}/{N_TEST:,})')
        continue
    t_c = time.time()
    raw_slice = test_df['text'].iloc[start:end].fillna('').astype(str).apply(preprocess_text).tolist()
    order = np.argsort([len(t) for t in raw_slice])
    sorted_slice = [raw_slice[i] for i in order]
    pa_s, pb_s, pc_s = predict_three_head(m_final_s7, sorted_slice, tok_final_s7, batch_size=256, max_len=256)
    inv_order = np.argsort(order)
    pa_c, pb_c, pc_c = pa_s[inv_order], pb_s[inv_order], pc_s[inv_order]
    np.savez(c_file, pa=pa_c, pb=pb_c, pc=pc_c)
    s7_pa_chunks.append(pa_c); s7_pb_chunks.append(pb_c); s7_pc_chunks.append(pc_c)
    dt = time.time() - t_c
    rps = len(raw_slice) / dt
    eta_min = ((N_TEST - end) / rps) / 60.0
    print(f'[XLM-R seed=7] Chunk {c_idx+1}: {end:,}/{N_TEST:,} | {rps:.1f} rows/s | ETA: {eta_min:.1f} min')

test_s7_pa = np.vstack(s7_pa_chunks)
test_s7_pb = np.vstack(s7_pb_chunks)
test_s7_pc = np.vstack(s7_pc_chunks)
print(f'XLM-R seed=7 total inference time: {(time.time()-t_s7_all)/60:.2f} min')

# Mean of the two XLM-R models, blended with TF-IDF
xlmr_mean_pa = 0.5 * (test_xlmr_pa + test_s7_pa)
xlmr_mean_pb = 0.5 * (test_xlmr_pb + test_s7_pb)
xlmr_mean_pc = 0.5 * (test_xlmr_pc + test_s7_pc)

v2_pa = wA * xlmr_mean_pa + (1.0 - wA) * test_tf_pa
v2_pb = wB * xlmr_mean_pb + (1.0 - wB) * test_tf_pb
v2_pc = wC * xlmr_mean_pc + (1.0 - wC) * test_tf_pc

v2_labels = np.argmax(v2_pa, axis=1)
v2_actions = np.argmax(v2_pc, axis=1)
v2_confs = calibrate_conf_v2(v2_pa)
v2_techs = []
v2_noise_hits = 0

for i in range(N_TEST):
    if is_oov_noise_only(clean_all_test[i]):
        v2_labels[i] = 0
        v2_techs.append('NONE')
        v2_actions[i] = 0
        v2_confs[i] = 0.5
        v2_noise_hits += 1
        continue
    lbl = int(v2_labels[i])
    if lbl in (0, 5):
        v2_techs.append('NONE')
        v2_actions[i] = 0
    else:
        active = [TECH_CODES[j] for j in range(6) if v2_pb[i, j] >= FINAL_TECH_THRESH[j]]
        if not active:
            active = [TECH_CODES[int(np.argmax(v2_pb[i]))]]
        v2_techs.append('|'.join(sorted(active)))

sub_v2_path = os.path.join(WORK_DIR, 'submission_v2.csv')
pd.DataFrame({
    'id': test_df['id'].values.astype(int),
    'label': v2_labels.astype(int),
    'techniques': v2_techs,
    'action': v2_actions.astype(int),
    'confidence': v2_confs,
}).to_csv(sub_v2_path, index=False)
print(f'Wrote {sub_v2_path} ({N_TEST:,} rows, noise_hits={v2_noise_hits:,})')

# Inline validator on submission_v2.csv
v2_chk = pd.read_csv(sub_v2_path, keep_default_na=False)
assert list(v2_chk.columns) == ['id', 'label', 'techniques', 'action', 'confidence'], 'Header mismatch'
assert len(v2_chk) == N_TEST == len(ref_ids), 'Row count mismatch'
assert (v2_chk['id'].astype(int).values == test_df['id'].values).all(), 'ID order mismatch'
assert not v2_chk['id'].duplicated().any(), 'Duplicate IDs'
for col in v2_chk.columns:
    assert (v2_chk[col].astype(str).str.strip() != '').all(), f'Blank values in {col}'
assert v2_chk['label'].astype(int).isin([0,1,2,3,4,5]).all(), 'Invalid label'
assert v2_chk['action'].astype(int).isin([0,1,2,3,4,5]).all(), 'Invalid action'
assert v2_chk['techniques'].apply(_chk_tech).all(), 'Invalid techniques'
c_v2 = pd.to_numeric(v2_chk['confidence'], errors='raise')
assert not c_v2.isna().any() and ((c_v2 >= 0.0) & (c_v2 <= 1.0)).all(), 'Invalid confidence'
print('PASS: submission_v2.csv passed all validator checks!')

# Compare with submission_final.csv
fin_chk = pd.read_csv(sub_final_path, keep_default_na=False)
fin_lbls = fin_chk['label'].astype(int).values
v2_lbls = v2_chk['label'].astype(int).values
n_changed = int((fin_lbls != v2_lbls).sum())
agree_rate = (fin_lbls == v2_lbls).mean() * 100.0
print(f'\nLabel changes vs submission_final.csv: {n_changed:,} / {N_TEST:,} rows | Agreement rate: {agree_rate:.2f}%')

print('\nPredicted Label Distribution (submission_v2.csv vs submission_final.csv):')
vc_v2 = pd.Series(v2_lbls).value_counts().sort_index()
vc_fn = pd.Series(fin_lbls).value_counts().sort_index()
for k in range(6):
    c2, cf = int(vc_v2.get(k, 0)), int(vc_fn.get(k, 0))
    print(f'  Label {k}: v2={c2:7,d} ({c2/N_TEST*100:5.2f}%)  | final={cf:7,d} ({cf/N_TEST*100:5.2f}%)')

print('\nFiles in /kaggle/working:')
for fn in sorted(os.listdir(WORK_DIR)):
    print(' ', fn)
